In [1]:
# GPU'yu kontrol et
import torch
print(torch.cuda.is_available())  # True çıkmalı
print(torch.cuda.get_device_name(0))  # GPU adı

True
Tesla T4


In [2]:
# Google Drive'ı bağla (veri kaybolmasın diye)
from google.colab import drive
drive.mount('/content/drive')

# Çalışma klasörü oluştur
import os
os.makedirs('/content/drive/MyDrive/SkinXAI', exist_ok=True)
os.makedirs('/content/drive/MyDrive/SkinXAI/data', exist_ok=True)
os.makedirs('/content/drive/MyDrive/SkinXAI/models', exist_ok=True)


Mounted at /content/drive


In [ ]:
!pip install timm albumentations opencv-python-headless kaggle

In [4]:
import torch #tensor işlemleri ve GPU desteği
import torch.nn as nn #sinir ağı katmanları içerir
import torch.nn.functional as F #aktivasyon, loss fonksiyonları
from torch.utils.data import Dataset, DataLoader #veri seti oluşturma
import timm #EfficientNet, ImageNet
import albumentations as A #data augumantation
from albumentations.pytorch import ToTensorV2 # Albumentations çıktılarını PyTorch tensor formatına çeviren araç.
import pandas as pd #csv dosyaları yönetir
import numpy as np
import matplotlib.pyplot as plt # Görüntüleri görselleştirmek ve eğitim grafiklerini çizmek için.
import cv2 # Görüntü okuma (imread), boyutlandırma ve renk uzayı dönüşümleri (OpenCV).
import os # Dosya yolları ve klasör işlemleri (dosya var mı kontrolü vb.) için.
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [22]:
# 1. Yeni token yükle (emin olmak için)
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 2. Doğru isimle indir
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 \
    -p /content/drive/MyDrive/SkinXAI/data/ham10000 \
    --unzip

# 3. Kontrol et
!ls /content/drive/MyDrive/SkinXAI/data/ham10000/

Saving kaggle.json to kaggle (1).json
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [00:50<00:00, 110MB/s]

ham10000_images_part_1	HAM10000_images_part_2	hmnist_28_28_RGB.csv
HAM10000_images_part_1	HAM10000_metadata.csv	hmnist_8_8_L.csv
ham10000_images_part_2	hmnist_28_28_L.csv	hmnist_8_8_RGB.csv


In [28]:
!kaggle datasets download -d cdeotte/jpeg-isic2019-384x384 \
    -p /content/drive/MyDrive/SkinXAI/data/isic2020 \
    --unzip

!ls /content/drive/MyDrive/SkinXAI/data/isic2020/

Dataset URL: https://www.kaggle.com/datasets/cdeotte/jpeg-isic2019-384x384
License(s): CC0-1.0
100% 855M/855M [00:12<00:00, 70.6MB/s]

train  train.csv


In [36]:
'''# PAD-UFES-20 Kaggle versiyonu
!kaggle datasets download -d mahdavi1202/skin-cancer-dataset \
    -p /content/drive/MyDrive/SkinXAI/data/pad_ufes \
    --unzip

!ls /content/drive/MyDrive/SkinXAI/data/pad_ufes/'''

'# PAD-UFES-20 Kaggle versiyonu\n!kaggle datasets download -d mahdavi1202/skin-cancer-dataset     -p /content/drive/MyDrive/SkinXAI/data/pad_ufes     --unzip\n\n!ls /content/drive/MyDrive/SkinXAI/data/pad_ufes/'

In [34]:
# HAM10000 sınıf dağılımı
import pandas as pd

ham_df = pd.read_csv('/content/drive/MyDrive/SkinXAI/data/ham10000/HAM10000_metadata.csv')
print("=== HAM10000 ===")
print(ham_df['dx'].value_counts())
print(f"Toplam: {len(ham_df)}")

# ISIC 2019 sınıf dağılımı
isic_df = pd.read_csv('/content/drive/MyDrive/SkinXAI/data/isic2020/train.csv')
print("\n=== ISIC 2019 ===")
print(isic_df['diagnosis'].value_counts())
print(f"Toplam: {len(isic_df)}")

=== HAM10000 ===
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
Toplam: 10015

=== ISIC 2019 ===
diagnosis
NV      12875
MEL      4522
BCC      3323
BKL      2624
AK        867
SCC       628
VASC      253
DF        239
Name: count, dtype: int64
Toplam: 25331


In [35]:
import os

# HAM10000 hazırla
ham_df = pd.read_csv('/content/drive/MyDrive/SkinXAI/data/ham10000/HAM10000_metadata.csv')

# Görüntü yollarını bul
def find_ham_image(image_id):
    for folder in ['ham10000_images_part_1', 'ham10000_images_part_2']:
        path = f'/content/drive/MyDrive/SkinXAI/data/ham10000/{folder}/{image_id}.jpg'
        if os.path.exists(path):
            return path
    return None

ham_df['image_path'] = ham_df['image_id'].apply(find_ham_image)
ham_df = ham_df.rename(columns={'dx': 'label'})
ham_df = ham_df[['image_path', 'label']]

# ISIC 2019 hazırla
isic_df = pd.read_csv('/content/drive/MyDrive/SkinXAI/data/isic2020/train.csv')
isic_df['image_path'] = isic_df['image_name'].apply(
    lambda x: f'/content/drive/MyDrive/SkinXAI/data/isic2020/train/{x}.jpg'
)
isic_df = isic_df.rename(columns={'diagnosis': 'label'})
isic_df['label'] = isic_df['label'].str.lower()
isic_df = isic_df[['image_path', 'label']]

# Etiket standardizasyonu
label_mapping = {
    'mel': 'melanoma', 'MEL': 'melanoma',
    'nv': 'nevus',     'NV': 'nevus',
    'bcc': 'bcc',      'BCC': 'bcc',
    'bkl': 'bkl',      'BKL': 'bkl',
    'akiec': 'ak',     'AK': 'ak',
    'vasc': 'vasc',    'VASC': 'vasc',
    'df': 'df',        'DF': 'df',
    'scc': 'scc',      'SCC': 'scc',
}

ham_df['label'] = ham_df['label'].map(label_mapping)
isic_df['label'] = isic_df['label'].map(label_mapping)

# Birleştir
all_df = pd.concat([ham_df, isic_df], ignore_index=True)
all_df = all_df.dropna()

print(all_df['label'].value_counts())
print(f"\nToplam: {len(all_df)}")
print(f"Eksik görüntü: {all_df['image_path'].isna().sum()}")

label
nevus       19580
melanoma     5635
bcc          3837
bkl          3723
scc           628
vasc          395
df            354
ak            327
Name: count, dtype: int64

Toplam: 34479
Eksik görüntü: 0


In [37]:
from sklearn.model_selection import train_test_split

# Etiketleri sayıya çevir
CLASS_NAMES = sorted(all_df['label'].unique())
print("Sınıflar:", CLASS_NAMES)

label_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
idx_to_label = {idx: name for name, idx in label_to_idx.items()}

all_df['label_idx'] = all_df['label'].map(label_to_idx)
print(all_df.head())

# Train / Val / Test böl
train_val_df, test_df = train_test_split(
    all_df,
    test_size=0.2,
    stratify=all_df['label'],
    random_state=42
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.2,
    stratify=train_val_df['label'],
    random_state=42
)

print(f"\nTrain: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")
print(f"\nTrain sınıf dağılımı:")
print(train_df['label'].value_counts())

Sınıflar: ['ak', 'bcc', 'bkl', 'df', 'melanoma', 'nevus', 'scc', 'vasc']
                                          image_path label  label_idx
0  /content/drive/MyDrive/SkinXAI/data/ham10000/h...   bkl          2
1  /content/drive/MyDrive/SkinXAI/data/ham10000/h...   bkl          2
2  /content/drive/MyDrive/SkinXAI/data/ham10000/h...   bkl          2
3  /content/drive/MyDrive/SkinXAI/data/ham10000/h...   bkl          2
4  /content/drive/MyDrive/SkinXAI/data/ham10000/h...   bkl          2

Train: 22066
Val:   5517
Test:  6896

Train sınıf dağılımı:
label
nevus       12531
melanoma     3606
bcc          2456
bkl          2382
scc           402
vasc          253
df            226
ak            210
Name: count, dtype: int64


In [38]:
class SkinDataset(Dataset):
    def __init__(self, df, is_train=True):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

        # Az olan sınıflar
        self.minority_classes = ['df', 'ak', 'vasc', 'scc']
        self.medium_classes = ['melanoma', 'bcc', 'bkl']

    def get_transform(self, label):
        if not self.is_train:
            # Val/Test → sadece resize ve normalize
            return A.Compose([
                A.Resize(224, 224),
                A.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                ),
                ToTensorV2()
            ])

        if label in self.minority_classes:
            # Az veri → çok agresif augmentation
            return A.Compose([
                A.Resize(224, 224),
                A.HorizontalFlip(p=0.8),
                A.VerticalFlip(p=0.8),
                A.Rotate(limit=180, p=0.8),
                A.RandomBrightnessContrast(p=0.7),
                A.HueSaturationValue(p=0.5),
                A.ElasticTransform(p=0.4),
                A.GridDistortion(p=0.4),
                A.GaussianBlur(p=0.3),
                A.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                ),
                ToTensorV2()
            ])

        elif label in self.medium_classes:
            # Orta miktarda augmentation
            return A.Compose([
                A.Resize(224, 224),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.Rotate(limit=90, p=0.5),
                A.RandomBrightnessContrast(p=0.3),
                A.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                ),
                ToTensorV2()
            ])

        else:
            # nevus → minimal augmentation (zaten çok var)
            return A.Compose([
                A.Resize(224, 224),
                A.HorizontalFlip(p=0.3),
                A.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                ),
                ToTensorV2()
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = cv2.imread(row['image_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Sınıfa göre transform seç
        transform = self.get_transform(row['label'])
        img = transform(image=img)['image']

        return img, int(row['label_idx'])

# Dataset oluştur
train_dataset = SkinDataset(train_df, is_train=True)
val_dataset   = SkinDataset(val_df, is_train=False)
test_dataset  = SkinDataset(test_df, is_train=False)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=32,
                          shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32,
                          shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32,
                          shuffle=False, num_workers=2)

# Test et
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")
print("DataLoader hazır! ✅")

Batch shape: torch.Size([32, 3, 224, 224])
DataLoader hazır! ✅


In [39]:
import timm

class SkinXAI_Model(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        # EfficientNet-B4 yükle (ImageNet ağırlıkları ile)
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained=True,
            num_classes=0,   # Son katmanı çıkar
            global_pool=''   # Pooling'i çıkar
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(1792, num_classes)

    def forward(self, x):
        features = self.backbone(x)        # Özellik çıkar
        pooled = self.pool(features).flatten(1)  # Düzleştir
        pooled = self.dropout(pooled)      # Dropout
        return self.classifier(pooled)     # Sınıflandır

# Modeli oluştur
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Cihaz: {device}")

model = SkinXAI_Model(num_classes=8).to(device)

# Parametre sayısı
total = sum(p.numel() for p in model.parameters())
print(f"Toplam parametre: {total:,}")

Cihaz: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Toplam parametre: 17,562,960


In [40]:
from sklearn.utils.class_weight import compute_class_weight

# Class weights hesapla
weights = compute_class_weight(
    'balanced',
    classes=np.arange(8),
    y=train_df['label_idx'].values
)
weights_tensor = torch.FloatTensor(weights).to(device)

print("Class weights:")
for i, (name, w) in enumerate(zip(CLASS_NAMES, weights)):
    print(f"  {name}: {w:.2f}")

# Loss (class weighted)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

# AŞAMA 1: Sadece classifier eğit (backbone dondur)
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam(
    model.classifier.parameters(), lr=1e-3
)

print("\nAşama 1 hazır!")
print("Backbone: Donduruldu ❄️")
print("Classifier: Eğitiliyor 🔥")

Class weights:
  ak: 13.13
  bcc: 1.12
  bkl: 1.16
  df: 12.20
  melanoma: 0.76
  nevus: 0.22
  scc: 6.86
  vasc: 10.90

Aşama 1 hazır!
Backbone: Donduruldu ❄️
Classifier: Eğitiliyor 🔥


In [41]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += len(labels)

        # Her 100 batch'te bir yazdır
        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(loader)} | "
                  f"Loss: {loss.item():.3f}")

    return total_loss/len(loader), correct/total


def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += len(labels)

    return total_loss/len(loader), correct/total


# Eğitimi başlat
print("AŞAMA 1 Eğitimi Başlıyor...")
print("="*50)

best_val_acc = 0
history = {'train_loss': [], 'val_loss': [],
           'train_acc': [], 'val_acc': []}

for epoch in range(10):
    print(f"\nEpoch {epoch+1}/10")

    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = val_epoch(
        model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f"Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.3f}")
    print(f"Val Loss:   {val_loss:.3f} | Val Acc:   {val_acc:.3f}")

    # En iyi modeli kaydet
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            model.state_dict(),
            '/content/drive/MyDrive/SkinXAI/models/best_model.pth'
        )
        print(f"✅ Model kaydedildi! Best Val Acc: {best_val_acc:.3f}")

print("\nAşama 1 tamamlandı!")
print(f"En iyi Val Accuracy: {best_val_acc:.3f}")

AŞAMA 1 Eğitimi Başlıyor...

Epoch 1/10
  Batch 0/690 | Loss: 2.127
  Batch 100/690 | Loss: 1.588
  Batch 200/690 | Loss: 1.586
  Batch 300/690 | Loss: 1.397
  Batch 400/690 | Loss: 1.540
  Batch 500/690 | Loss: 1.469
  Batch 600/690 | Loss: 1.315
Train Loss: 1.584 | Train Acc: 0.622
Val Loss:   1.592 | Val Acc:   0.627
✅ Model kaydedildi! Best Val Acc: 0.627

Epoch 2/10
  Batch 0/690 | Loss: 1.325
  Batch 100/690 | Loss: 1.001
  Batch 200/690 | Loss: 1.096
  Batch 300/690 | Loss: 1.468
  Batch 400/690 | Loss: 1.086
  Batch 500/690 | Loss: 1.262
  Batch 600/690 | Loss: 1.428
Train Loss: 1.330 | Train Acc: 0.636
Val Loss:   1.500 | Val Acc:   0.640
✅ Model kaydedildi! Best Val Acc: 0.640

Epoch 3/10
  Batch 0/690 | Loss: 0.727
  Batch 100/690 | Loss: 1.056
  Batch 200/690 | Loss: 1.318
  Batch 300/690 | Loss: 1.423
  Batch 400/690 | Loss: 1.204
  Batch 500/690 | Loss: 0.975
  Batch 600/690 | Loss: 1.378
Train Loss: 1.270 | Train Acc: 0.646
Val Loss:   1.445 | Val Acc:   0.645
✅ Model ka

In [ ]:
print("AŞAMA 2: Fine-tuning başlıyor...")

# En iyi modeli yükle
model.load_state_dict(
    torch.load('/content/drive/MyDrive/SkinXAI/models/best_model.pth')
)

# Tüm katmanları aç
for param in model.parameters():
    param.requires_grad = True

# Farklı learning rate
optimizer = torch.optim.Adam([
    {'params': model.backbone.parameters(), 'lr': 1e-5},   # Yavaş
    {'params': model.classifier.parameters(), 'lr': 1e-4}  # Hızlı
])

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=20
)

best_val_acc = 0.662  # Aşama 1'in en iyisi

for epoch in range(20):
    print(f"\nEpoch {epoch+1}/20")

    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = val_epoch(
        model, val_loader, criterion, device)

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f"Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.3f}")
    print(f"Val Loss:   {val_loss:.3f} | Val Acc:   {val_acc:.3f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            model.state_dict(),
            '/content/drive/MyDrive/SkinXAI/models/best_model.pth'
        )
        print(f"✅ Model kaydedildi! Best Val Acc: {best_val_acc:.3f}")

print(f"\nAşama 2 tamamlandı!")
print(f"En iyi Val Accuracy: {best_val_acc:.3f}")